# 1. 단일 기업 재무제표 받기

In [1]:
import sys
import pandas as pd
import time

# 1. 모듈 임포트 및 경로 설정
sys.path.append(r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\collect")

from sec_data_pipeline.collectors.get_us_ticker import get_filtered_us_tickers
from sec_data_pipeline.collectors.rate_limiter import AdaptiveRateLimiter # 반드시 확인
from sec_data_pipeline.valuation.integrated_financial_analyzer_mysql_fixed import IntegratedFinancialAnalyzer
from sec_data_pipeline.storage.db_manager import DBManager
from DATA.stock_invest_function import get_db_host

# ---------------------------------------------------------
# [설정 항목] 여기서 날짜와 개수를 조절하세요
# ---------------------------------------------------------
START_DATE = "2025-01-01"  # ✅ 이 날짜 이후의 데이터만 수집/저장
MAX_TICKERS = 5000         # ✅ 테스트로 몇 개만 할지 결정 (전체는 None 또는 큰 숫자)
OFFSET = 0              # ✅ 시작 위치 (4000번 인덱스부터)
# ---------------------------------------------------------

# 2. DB 및 객체 초기화
db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}
db_manager = DBManager(db_info)
analyzer = IntegratedFinancialAnalyzer(db_info=db_info, wrds_conn_str="여기에_WRDS_주소")

# ✅ [중요] NameError 방지를 위한 rate_limiter 정의
rate_limiter = AdaptiveRateLimiter(
    max_calls=8,
    time_window=1.0,
    min_calls=3,
    backoff_factor=0.8,
)

headers = {"User-Agent": "Hoyoung Research <stox1224@gmail.com>"}

# 3. 티커 리스트 준비 (슬라이싱 적용)
ALL_TICKERS = get_filtered_us_tickers()
TICKER_LIST = ALL_TICKERS[OFFSET : OFFSET + MAX_TICKERS]

TICKER_LIST = ['GOOG']

print(f"총 {len(TICKER_LIST)}개의 티커를 수집합니다. (기준일: {START_DATE})")

# 4. 메인 루프
for i, ticker in enumerate(TICKER_LIST, start=1):
    print(f"\n=== [{i}/{len(TICKER_LIST)}] {ticker} 처리 시작 ===")

    # 이제 정의된 rate_limiter를 사용하므로 에러가 나지 않습니다.
    rate_limiter.wait_if_needed()

    try:
        result = analyzer.analyze(ticker, headers=headers, table_name="us_fundq")
        if result is None:
            continue

        final_df, cik, entity_name = result

        # ✅ [추가] 날짜 필터링 로직 적용
        if not final_df.empty:
            final_df.index = pd.to_datetime(final_df.index)
            # 설정한 START_DATE 이후 데이터만 필터링
            final_df = final_df[final_df.index >= pd.to_datetime(START_DATE)]

        if final_df.empty:
            print(f"⚠ {ticker}: {START_DATE} 이후의 새로운 데이터가 없어 저장을 건너뜁니다.")
            continue

        # DB 저장용 인덱스 정리
        if final_df.index.name != "date":
            final_df.index.name = "date"

        db_manager.save_normalized_data(
            ticker=ticker,
            cik=cik,
            df=final_df,
            item_mapping=None
        )
        print(f"✓ {ticker} ({entity_name}) {len(final_df)}건 저장 완료")

    except Exception as e:
        msg = str(e)
        if "429" in msg:
            print(f"⚠ {ticker}: SEC 429 감지 → 60초 대기")
            rate_limiter.on_rate_limit_error()
            time.sleep(60)
            continue
        print(f"✗ {ticker} 처리 중 오류: {e}")
        continue

print("\n=== 작업 완료 ===")


✓ DB connected: 192.168.0.230:3307/investar


100%|██████████| 308/308 [00:00<00:00, 877.57it/s] 


제외된 기업 수: 1802
남은 기업 수: 5081
티커 수: 5081
총 1개의 티커를 수집합니다. (기준일: 2025-01-01)

=== [1/1] GOOG 처리 시작 ===
[GOOG] 통합 재무분석 시작
✓ Entity: ALPHABET INC. (CIK: 1652044)
✓ EDGAR 정규화 DF: 48 rows × 31 cols
✓ MySQL 연결 성공: 192.168.0.230:3307/investar
⚠ WRDS 쿼리 실패: Execution failed on sql 'SELECT * FROM us_fundq WHERE tic=%s': (1146, "Table 'investar.us_fundq' doesn't exist")
WRDS 데이터 없음 → EDGAR 그대로 반환
✓ EDGAR+WRDS 병합 DF: 48 rows × 31 cols
재무비율 계산 중...
  - 수익성 비율 계산...
  - 레버리지 비율 계산...
  - 유동성 비율 계산...
  - 효율성 비율 계산...
재무비율 계산 완료!
✓ 최종 DF (비율 포함): 48 rows × 47 cols
[GOOG] 통합 재무분석 종료
✓ Saved GOOG data: 4 dates x 47 items
✓ GOOG (ALPHABET INC.) 4건 저장 완료

=== 작업 완료 ===


In [3]:
final_df

,revenue,net_income,operating_income,total_assets,current_assets,total_liabilities,current_liabilities,stockholders_equity,cash,long_term_debt,...,debt_to_equity,debt_to_assets,equity_multiplier,current_ratio,quick_ratio,inventory_turnover,days_inventory,receivables_turnover,days_receivables,asset_turnover
date,,,,,,,,,,,,,,,,,,,,,
2025-03-31,9.023400e+10,3.454000e+10,3.060600e+10,4.753740e+11,1.620520e+11,1.301070e+11,9.165400e+10,3.452670e+11,2.326400e+10,1.200000e+10,...,37.683011,27.369398,1.376830,1.768084,1.735822,12.296584,29.683040,1.888689,193.255757,0.204444
2025-06-30,9.023400e+10,2.819600e+10,3.127100e+10,5.020530e+11,1.662160e+11,1.391370e+11,8.731000e+10,3.629160e+11,2.103600e+10,1.200000e+10,...,38.338624,27.713608,1.383386,1.903745,1.869877,13.202232,27.646840,1.766956,206.570001,0.196841
2025-09-30,9.023400e+10,3.497900e+10,3.122800e+10,5.364690e+11,1.739470e+11,1.496020e+11,9.955000e+10,3.868670e+11,2.309000e+10,1.200000e+10,...,38.670137,27.886420,1.386701,1.747333,1.717629,13.990193,26.089705,1.698490,214.896713,0.186678
2025-12-31,9.023400e+10,3.497900e+10,3.122800e+10,5.952810e+11,2.060380e+11,1.800160e+11,1.027450e+11,4.152650e+11,3.070800e+10,4.908500e+10,...,43.349668,30.240508,1.433497,2.005334,1.976554,13.990193,26.089705,1.566209,233.046801,0.172608


In [9]:
import pandas as pd
import pymysql
from typing import Dict, Optional, List


def quarter_end_from_report_date(report_date: pd.Timestamp) -> pd.Timestamp:
    y = report_date.year
    m = report_date.month

    if m in [1, 2, 3]:
        return pd.Timestamp(f"{y}-03-31")
    elif m in [4, 5, 6]:
        return pd.Timestamp(f"{y}-06-30")
    elif m in [7, 8, 9]:
        return pd.Timestamp(f"{y}-09-30")
    else:
        return pd.Timestamp(f"{y}-12-31")


def fetch_sec_financial_pivot_fixed(
    db_info: Dict[str, any],
    ticker: str,
    table_name: str = "sec_financial_data",
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    item_list: Optional[List[str]] = None,
) -> pd.DataFrame:

    conn = pymysql.connect(
        host=db_info["host"], port=db_info.get("port", 3306),
        user=db_info["user"], password=db_info["password"],
        database=db_info["database"], charset="utf8mb4"
    )

    # -----------------------------
    # 1) 데이터 로드
    # -----------------------------
    try:
        where_clause = ["ticker=%s"]
        params = [ticker]

        if start_date:
            where_clause.append("date >= %s")
            params.append(start_date)
        if end_date:
            where_clause.append("date <= %s")
            params.append(end_date)
        if item_list:
            placeholders = ",".join(["%s"] * len(item_list))
            where_clause.append(f"item_name IN ({placeholders})")
            params.extend(item_list)

        where_sql = " AND ".join(where_clause)

        sql = f"""
            SELECT date, ticker, item_name, value
            FROM {table_name}
            WHERE {where_sql}
            ORDER BY date, item_name
        """

        df = pd.read_sql(sql, conn, params=params)
    finally:
        conn.close()

    if df.empty:
        print(f"[INFO] {ticker} 데이터 없음")
        return df

    # -----------------------------
    # 2) 기존 date → report_date로 이름 변경
    # -----------------------------
    df = df.rename(columns={"date": "report_date"})
    df["report_date"] = pd.to_datetime(df["report_date"])

    # -----------------------------
    # 3) report_date → 분기말 날짜(date)
    # -----------------------------
    df["date"] = df["report_date"].apply(quarter_end_from_report_date)

    # (중복 가능성 대비) 각 분기(date, ticker)별 대표 report_date 하나 선택
    # 여기서는 가장 늦은 날짜(max) 사용
    rep_dates = (
        df[["date", "ticker", "report_date"]]
        .drop_duplicates()
        .groupby(["date", "ticker"], as_index=False)["report_date"]
        .max()
    )

    # -----------------------------
    # 4) pivot 변환 (분기말 기준 wide)
    # -----------------------------
    df_pivot = df.pivot_table(
        index=["date", "ticker"],
        columns="item_name",
        values="value",
        aggfunc="last"
    ).sort_index()

    df_pivot = df_pivot.reset_index()

    # -----------------------------
    # 5) 대표 report_date 다시 붙이기
    # -----------------------------
    df_pivot = df_pivot.merge(rep_dates, how="left", on=["date", "ticker"])

    # 보기 편하게 컬럼 순서 조정: date, report_date, ticker, 나머지 item 들
    cols = ["ticker"] + [c for c in df_pivot.columns if c not in ("ticker")]
    df_pivot = df_pivot[cols]
    df_pivot = df_pivot[cols]

    return df_pivot


In [17]:
# 1) AAPL 전체 기간, 모든 item_name
fs_df = fetch_sec_financial_pivot_fixed(db_info, "POOL")
fs_df.tail(5)

# 2) 기간 제한 + 특정 항목만
# item_list = ["revenue", "net_income", "total_assets", "ROE", "OPM"]
# aapl_subset = fetch_sec_financial_pivot(
#     db_info,
#     "AAPL",
#     start_date="2015-01-01",
#     end_date="2024-12-31",
#     item_list=item_list
# )
# print(aapl_subset.tail())

,ticker,date,accounts_payable,accounts_receivable,accrued_liabilities,accumulated_depreciation,asset_turnover,capital_expenditures,cash,cost_of_revenue,...,receivables_turnover,revenue,roa,roe,roic,short_term_debt,stockholders_equity,total_assets,total_liabilities,report_date
63,POOL,2024-09-30,401702000.0,119538000.0,185118000.0,233136000.0,0.311205,17038000.0,36693000.0,1.016480e+09,...,8.12024,1.057800e+09,13.1937,31.4629,5.75478,44683000.0,1.432510e+09,3.367390e+09,1.934880e+09,2024-09-30
64,POOL,2024-12-31,525235000.0,115835000.0,171194000.0,255154000.0,0.311290,17038000.0,36693000.0,1.016480e+09,...,8.08848,1.057800e+09,15.3828,40.4234,5.99564,49473000.0,1.273460e+09,3.368180e+09,2.094720e+09,2024-12-31
65,POOL,2025-03-31,890167000.0,146209000.0,109893000.0,255154000.0,0.283166,13295000.0,36693000.0,7.591570e+08,...,7.13650,1.057800e+09,13.3146,38.4918,2.77570,57059000.0,1.238690e+09,3.712450e+09,2.473760e+09,2025-03-31
66,POOL,2025-06-30,529316000.0,172028000.0,160833000.0,255154000.0,0.289299,13295000.0,36693000.0,1.249370e+09,...,6.18821,1.057800e+09,13.6528,36.6687,8.14728,17386000.0,1.299120e+09,3.671960e+09,2.372840e+09,2025-06-30
67,POOL,2025-09-30,457319000.0,138072000.0,147122000.0,255154000.0,0.308036,13295000.0,36693000.0,1.021950e+09,...,8.21245,1.057800e+09,14.5752,35.5936,5.76868,12881000.0,1.379890e+09,3.500670e+09,2.120780e+09,2025-09-30


In [11]:
# item_list = ["revenue", "net_income", "total_assets", "ROE", "OPM"]
# aapl_subset = fetch_sec_financial_pivot(
#     db_info,
#     "AAPL",
#     start_date="2015-01-01",
#     end_date="2025-12-31",
#     item_list=item_list
# )
# print(aapl_subset.tail())

NameError: name 'fetch_sec_financial_pivot' is not defined

In [6]:
def count_unique_tickers_sec_financial(db_info: dict,
                                       table_name: str = "sec_financial_data") -> int:
    """
    sec_financial_data 테이블에서 unique ticker 개수를 조회하여 반환하는 함수.
    """
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        query = f"SELECT DISTINCT ticker FROM {table_name};"
        df = pd.read_sql(query, conn)

        unique_ticker_count = df["ticker"].nunique()

        print(f"[INFO] Unique tickers in {table_name}: {unique_ticker_count}")
        return unique_ticker_count

    finally:
        conn.close()

In [7]:
import pymysql

count_unique_tickers_sec_financial(db_info)

[INFO] Unique tickers in sec_financial_data: 4060


4060

In [25]:
aapl_df.columns.tolist()

['date',
 'ticker',
 'accounts_payable',
 'accounts_receivable',
 'accrued_liabilities',
 'accumulated_depreciation',
 'asset_turnover',
 'capital_expenditures',
 'cash',
 'cost_of_revenue',
 'current_assets',
 'current_liabilities',
 'current_ratio',
 'days_inventory',
 'days_receivables',
 'debt_to_assets',
 'debt_to_equity',
 'deferred_revenue',
 'depreciation_amortization',
 'equity_multiplier',
 'goodwill',
 'gross_margin',
 'gross_profit',
 'income_tax_expense',
 'intangible_assets',
 'interest_expense',
 'interest_income',
 'inventory',
 'inventory_turnover',
 'long_term_debt',
 'long_term_debt_current',
 'net_income',
 'net_margin',
 'net_ppe',
 'operating_expenses',
 'operating_income',
 'operating_margin',
 'other_noncurrent_assets',
 'pretax_income',
 'quick_ratio',
 'receivables_turnover',
 'research_development',
 'revenue',
 'roa',
 'roe',
 'roic',
 'short_term_debt',
 'stockholders_equity',
 'total_assets',
 'total_liabilities']

In [ ]:
# # 1) EDGAR 수집/정규화
# headers = {"User-Agent": "HoyoungPark Research <stox1224@email.com>"}
# facts   = fetch_company_facts( TICKER, headers=headers)
# parser  = CompanyFactsParser(facts)
# normal  = FinancialNormalizer(parser)
# edgar_df = normal.create_normalized_dataframe(period_type="quarterly")
#
# # 2) WRDS Validator
# db_info = {
#     "host": get_db_host(),
#     "port": 3307,
#     "user": "stox7412",
#     "password": "Apt106503!~",
#     "database": "investar",
# }
# validator = WRDSDataValidator(db_info)
#
# import sqlalchemy as sa
#
# # ① EDGAR 분기 DF 준비 (이미 갖고 계신 df: edgar_df)
# #    edgar_df.index = 분기 날짜, 컬럼에 'revenue' 포함
#
# # ② WRDS(us_fundq)에서 AAPL의 edate, ticker, saleq 로드
# db_info = {
#     "host": "192.168.0.230",
#     "port": 3307,
#     "user": "stox7412",
#     "password": "Apt106503!~",
#     "database": "investar",
# }


In [3]:
tickers = get_filtered_us_tickers()

100%|██████████| 294/294 [00:00<00:00, 1034.88it/s]


제외된 기업 수: 1812
남은 기업 수: 5001
티커 수: 5001


In [5]:
ticker_list = tickers[:10]

['NVDA',
 'AAPL',
 'MSFT',
 'AMZN',
 'GOOGL',
 'AVGO',
 'GOOG',
 'META',
 'TSLA',
 'NFLX']

In [6]:
# import pandas as pd
#
# # 1. 클라이언트 및 다운로더 설정
# user_agent = "PersonalResearch stox1224@email.com"
# client = SECAPIClient(user_agent, RateLimiter(10, 1.0))
# downloader = BulkDownloader(client, output_dir="./sec_data", max_workers=3)
#
# # 2. 다운로드할 기업 리스트
# # tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'META']
#
# # 3. 배치 다운로드 (자동으로 JSON 파일 저장)
# results = downloader.download_company_facts_batch(ticker_list)
#
# # 4. 각 기업별 데이터 처리
# all_financials = {}
#
# for ticker, company_facts in results.items():
#     headers = {"User-Agent": "HoyoungPark Research <stox1224@email.com>"}
#     facts   = fetch_company_facts(ticker, headers=headers)
#     parser  = CompanyFactsParser(facts)
#     normal  = FinancialNormalizer(parser)
#     edgar_df = normal.create_normalized_dataframe(period_type="quarterly")
#
#
#     parser = CompanyFactsParser(company_facts)
#     normalizer = FinancialNormalizer(parser)
#
#     # 재무데이터 정규화
#     df = normalizer.create_normalized_dataframe(period_type='quarterly')
#     df_billions = normalizer.convert_to_billions(df)
#
#     all_financials[ticker] = df_billions
#
#     # 개별 CSV 저장
#     # normalizer.export_to_csv(df_billions, f'{ticker}_financials.csv')
#
# print(f"총 {len(all_financials)}개 기업 데이터 수집 완료")

NameError: name 'SECAPIClient' is not defined

In [21]:
TICKER = "GLW"   # ← 여기만 바꾸면 전체가 따라옵니

# 1) EDGAR 수집/정규화
headers = {"User-Agent": "HoyoungPark Research <stox1224@email.com>"}
facts   = fetch_company_facts( TICKER, headers=headers)
parser  = CompanyFactsParser(facts)
normal  = FinancialNormalizer(parser)
edgar_df = normal.create_normalized_dataframe(period_type="quarterly")

# 2) WRDS Validator
db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}
validator = WRDSDataValidator(db_info)

# 3) 결합/보완 실행 (핵심: table_name='us_fundq', date_col='edate')
result_df = validator.validate_and_fill_improved(
    edgar_df=edgar_df,
    ticker= TICKER,
    table_name="us_fundq",   # 당신 DB 테이블
    days_tolerance=15,
    verbose=True,
    ticker_col="ticker",     # 당신 DB 컬럼
    date_col="edate"         # 당신 DB 컬럼
)

import sqlalchemy as sa

# ① EDGAR 분기 DF 준비 (이미 갖고 계신 df: edgar_df)
#    edgar_df.index = 분기 날짜, 컬럼에 'revenue' 포함

# ② WRDS(us_fundq)에서 AAPL의 edate, ticker, saleq 로드
db_info = {
    "host": "192.168.0.230",
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}
df_wrds = fetch_wrds_fundq_sample(
    db_info, ticker=TICKER , columns=["edate", "ticker", "saleq"], table_name="US_fundq"
)

# ③ 결측 보완 + 단위 정합
filled_df = fill_revenue_with_wrds(
    edgar_df=edgar_df,
    wrds_df=df_wrds,
    wrds_value_col="saleq",     # WRDS의 분기 매출 컬럼명
    days_tolerance=20,          # 분기말 근접 허용일
    verbose=True
)

print("\n[CHECK] revenue NaN:")
print("  before:", edgar_df["revenue"].isna().sum())
print("  after :", filled_df["revenue"].isna().sum())

if filled_df.empty:
    print("⚠ filled_df가 비어 있습니다 → result_df를 target_df로 대체합니다.")
    target_df = result_df.copy()
else:
    print("✓ filled_df가 유효합니다 → target_df에 filled_df를 사용합니다.")
    target_df = filled_df.copy()

from sec_data_pipeline.parsers.financial_normalizer import FinancialNormalizer


# parser는 EDGAR 데이터를 읽어온 CompanyFactsParser 등의 인스턴스
normalizer = FinancialNormalizer(parser=None)  # parser가 필요 없으면 None으로 두세요


# 1) 기본 추정: income_tax_expense / pretax_income
est = (filled_df.get('income_tax_expense') / filled_df.get('pretax_income'))

# 2) 이상치/음수/무한대 정리
est = est.replace([np.inf, -np.inf], np.nan)
# 적자 구간(pretax_income <= 0)은 추정치 제거
est = est.mask((filled_df.get('pretax_income') <= 0), np.nan)
# 0~0.5로 클리핑(50% 이상은 보통 이상치)
est = est.clip(lower=0.0, upper=0.3)

# 3) 결측 보강: 최근 값으로 보간 + 기본값 대체
# est = est.ffill().fillna(0.21)

# filled_df['tax_rate'] = 0.21
ratio = normal.calculate_financial_ratios(target_df)


['NVDA',
 'AAPL',
 'MSFT',
 'AMZN',
 'GOOGL',
 'AVGO',
 'GOOG',
 'META',
 'TSLA',
 'NFLX',
 'COST',
 'PLTR',
 'ASML',
 'AMD',
 'CSCO',
 'AZN',
 'MU',
 'TMUS',
 'ISRG',
 'SHOP',
 'PEP',
 'AMAT',
 'LRCX',
 'LIN',
 'APP',
 'AMGN',
 'QCOM',
 'INTC',
 'PDD',
 'BKNG',
 'GILD',
 'KLAC',
 'TXN',
 'ARM',
 'ADBE',
 'PANW',
 'CRWD',
 'ADI',
 'SNY',
 'HON',
 'VRTX',
 'MELI',
 'ADP',
 'SBUX',
 'CMCSA',
 'NTES',
 'ORLY',
 'DASH',
 'REGN',
 'CDNS',
 'MAR',
 'SNPS',
 'CTAS',
 'MNST',
 'MDLZ',
 'MRVL',
 'ABNB',
 'CSX',
 'ADSK',
 'WDAY',
 'IDXX',
 'FTNT',
 'TRI',
 'ROST',
 'PYPL',
 'STX',
 'WBD',
 'ALNY',
 'ARGX',
 'DDOG',
 'PCAR',
 'WDC',
 'EA',
 'MSTR',
 'BKR',
 'NXPI',
 'ROP',
 'FER',
 'JD',
 'ZS',
 'FAST',
 'TCOM',
 'SYM',
 'TTWO',
 'INSM',
 'MPWR',
 'FANG',
 'AXON',
 'CCEP',
 'BIDU',
 'PAYX',
 'TEAM',
 'CPRT',
 'EBAY',
 'CTSH',
 'ONC',
 'KDP',
 'GEHC',
 'CRWV',
 'RYAAY',
 'KMB',
 'FISV',
 'NTRA',
 'SNDK',
 'UAL',
 'EXPE',
 'VRSK',
 'KHC',
 'CSGP',
 'ERIC',
 'VOD',
 'TSCO',
 'ODFL',
 'MCHP',
 'FSLR'

In [7]:
import time
import requests
import pandas as pd
from typing import Dict, List, Optional, Tuple

# -----------------------------
# SEC endpoints
# -----------------------------
SEC_TICKER_CIK_URL = "https://www.sec.gov/files/company_tickers.json"
SEC_COMPANYFACTS_URL = "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json"

FORM_PRIORITY = {"10-Q": 4, "10-K": 3, "10-Q/A": 2, "10-K/A": 1}
KEEP_FORMS = set(FORM_PRIORITY.keys())

# 손익(duration) 항목: quarterly vs YTD를 "기간 길이"로 구분
DURATION_COLS = {"revenue", "net_income", "operating_income"}

DEFAULT_TAGS = {
    # Income Statement (duration)
    "revenue": ["Revenue", "SalesRevenueNet", "Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"],
    "net_income": ["NetIncomeLoss", "ProfitLoss"],
    "operating_income": ["OperatingIncomeLoss"],

    # Balance Sheet (instant)
    "total_assets": ["Assets"],
    "current_assets": ["AssetsCurrent"],
    "total_liabilities": ["Liabilities"],
    "current_liabilities": ["LiabilitiesCurrent"],
    "equity": ["StockholdersEquity", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"],
}

# -----------------------------
# HTTP helper (User-Agent 필수)
# -----------------------------
def sec_get_json(url: str, user_agent: str, timeout: int = 30, sleep_sec: float = 0.15) -> dict:
    headers = {
        "User-Agent": user_agent,
        "Accept": "application/json",
        "Accept-Encoding": "gzip, deflate",
    }
    r = requests.get(url, headers=headers, timeout=timeout)
    r.raise_for_status()
    if sleep_sec:
        time.sleep(sleep_sec)
    return r.json()

def ticker_to_cik10(ticker: str, user_agent: str) -> str:
    data = sec_get_json(SEC_TICKER_CIK_URL, user_agent=user_agent)
    t = ticker.upper().strip()
    for _, v in data.items():
        if str(v.get("ticker", "")).upper() == t:
            return f"{int(v['cik_str']):010d}"
    raise ValueError(f"CIK not found for ticker={ticker}")

def fetch_companyfacts(ticker: str, user_agent: str) -> dict:
    cik10 = ticker_to_cik10(ticker, user_agent=user_agent)
    return sec_get_json(SEC_COMPANYFACTS_URL.format(cik10=cik10), user_agent=user_agent)

# -----------------------------
# parsing helpers
# -----------------------------
def _pick_unit_block(concept_obj: dict) -> Optional[Tuple[str, List[dict]]]:
    units = concept_obj.get("units", {})
    if not units:
        return None
    if "USD" in units:
        return ("USD", units["USD"])
    best_unit = max(units.keys(), key=lambda k: len(units.get(k, [])))
    return (best_unit, units[best_unit])

def _to_df(items: List[dict]) -> pd.DataFrame:
    df = pd.DataFrame(items).copy()
    if df.empty:
        return df
    for col in ["end", "start", "filed"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    if "val" in df.columns:
        df["val"] = pd.to_numeric(df["val"], errors="coerce")
    return df

def extract_best_concept(companyfacts: dict, tag_candidates: List[str]) -> pd.DataFrame:
    """
    후보 태그들 중:
    - USD unit 존재
    - 관측치 수(유효 val 수) 최대
    인 태그를 선택.
    """
    facts = companyfacts.get("facts", {}).get("us-gaap", {})
    best = None
    best_score = -1
    best_tag = None
    best_unit = None

    for tag in tag_candidates:
        if tag not in facts:
            continue
        unit_block = _pick_unit_block(facts[tag])
        if not unit_block:
            continue
        unit, items = unit_block
        df = _to_df(items)
        if df.empty:
            continue
        df = df.dropna(subset=["end", "val"])
        score = len(df)
        if score > best_score:
            best_score = score
            best = df
            best_tag = tag
            best_unit = unit

    if best is None:
        return pd.DataFrame()

    best = best.copy()
    best["tag"] = best_tag
    best["unit"] = best_unit
    return best

# -----------------------------
# cleaning / dedupe
# -----------------------------
def _keep_forms(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or "form" not in df.columns:
        return df
    return df[df["form"].isin(KEEP_FORMS)].copy()

def _prep_core(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    need = {"end", "val", "form", "filed"}
    if not need.issubset(df.columns):
        return pd.DataFrame()
    df = df.dropna(subset=["end", "val", "form", "filed"]).copy()
    df = _keep_forms(df)
    return df

def _dedupe_end_fy_fp(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()
    df["form_score"] = df["form"].map(FORM_PRIORITY).fillna(-1).astype(int)
    # (end,fy,fp) 기준으로 최신/우선 form 1개
    keys = [k for k in ["end", "fy", "fp"] if k in df.columns]
    if len(keys) == 3:
        df = df.sort_values(["end", "fy", "fp", "form_score", "filed"], ascending=[True, True, True, False, False])
        df = df.drop_duplicates(subset=["end", "fy", "fp"], keep="first")
    return df

def _dedupe_end(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()
    df["form_score"] = df["form"].map(FORM_PRIORITY).fillna(-1).astype(int)
    df = df.sort_values(["end", "form_score", "filed"], ascending=[True, False, False])
    df = df.drop_duplicates(subset=["end"], keep="first")
    return df

# -----------------------------
# duration: quarter vs YTD 선택 로직
# -----------------------------
def _duration_days(df: pd.DataFrame) -> pd.Series:
    if "start" not in df.columns:
        return pd.Series([pd.NA] * len(df), index=df.index)
    return (df["end"] - df["start"]).dt.days

def _select_quarter_duration_first(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    duration fact에서:
    - 분기값 후보: 기간이 대략 70~120일 (13주 분기)
    - 누적(YTD) 후보: 기간이 대략 150~330일

    반환: (quarter_like, ytd_like)
    """
    if df.empty:
        return df, df
    d = _duration_days(df)
    out = df.copy()
    out["dur_days"] = d

    # start 없는 경우: dur_days가 NA → 여기선 quarter_like로도 ytd_like로도 못 분류
    # (이 경우는 후단에서 fp/프레임 기반으로 처리하거나, 마지막 수단으로 diff)
    quarter_like = out[(out["dur_days"].notna()) & (out["dur_days"].between(70, 120))].copy()
    ytd_like = out[(out["dur_days"].notna()) & (out["dur_days"].between(150, 330))].copy()
    unknown = out[out["dur_days"].isna()].copy()

    # unknown은 별도 반환하지 않고, ytd_like에 붙여서 “diff 후보”로 취급 (보수적으로)
    ytd_like = pd.concat([ytd_like, unknown], ignore_index=True)

    return quarter_like, ytd_like

def _build_quarter_series_duration(df: pd.DataFrame) -> pd.Series:
    """
    duration 항목에 대해:
    1) 분기값(기간 70~120일) 있으면 그걸 우선 사용
    2) 없으면 YTD(또는 unknown)에서 fy 내 diff로 분기 복원
    """
    if df.empty:
        return pd.Series(dtype="float")

    # fy/fp 없으면 복원이 어렵다 → end 최신 1개씩만 쓰는 보수적 fallback
    if not {"fy", "fp"}.issubset(df.columns):
        df2 = _dedupe_end(df)
        return df2.sort_values(["end", "filed"]).groupby("end")["val"].last().sort_index()

    df = df.copy()
    df["fy"] = pd.to_numeric(df["fy"], errors="coerce")
    df = df.dropna(subset=["fy", "fp"])
    df["fy"] = df["fy"].astype(int)

    # 1) 분기값 우선
    quarter_like, ytd_like = _select_quarter_duration_first(df)
    if not quarter_like.empty:
        # 같은 분기(end) 중복 제거 후 end 기준 series
        q = _dedupe_end_fy_fp(quarter_like)
        q = _dedupe_end(q)
        return q.sort_values(["end", "filed"]).groupby("end")["val"].last().sort_index()

    # 2) YTD로 복원
    y = _dedupe_end_fy_fp(ytd_like)

    # Q1~Q3 누적에서 분기 diff
    q = y[y["fp"].isin(["Q1", "Q2", "Q3"])].copy()
    fy = y[y["fp"].isin(["FY"])].copy()

    if q.empty:
        # 그래도 없으면 end 기준으로만
        y2 = _dedupe_end(y)
        return y2.sort_values(["end", "filed"]).groupby("end")["val"].last().sort_index()

    q = q.sort_values(["fy", "end"])
    q["qval"] = q.groupby("fy")["val"].diff()
    q["qval"] = q["qval"].fillna(q["val"])
    q["val"] = q["qval"]
    q = q.drop(columns=["qval"])

    # Q4 = FY - YTD(Q3) (가능하면)
    if not fy.empty:
        fy2 = fy[["fy", "end", "val", "form", "filed"]].rename(columns={"val": "fy_val"})
        ytd_q3 = y[y["fp"] == "Q3"][["fy", "val"]].rename(columns={"val": "ytd_q3"})
        merged = fy2.merge(ytd_q3, on="fy", how="left")

        qsum = q.groupby("fy")["val"].sum().rename("qsum").reset_index()
        merged = merged.merge(qsum, on="fy", how="left")

        merged["q4"] = merged["fy_val"] - merged["ytd_q3"]
        merged.loc[merged["q4"].isna(), "q4"] = merged["fy_val"] - merged["qsum"]

        q4 = merged[["fy", "end", "q4", "form", "filed"]].copy()
        q4["val"] = q4["q4"]
        q4 = q4.drop(columns=["q4"])
        q = pd.concat([q[["end", "val", "form", "filed"]], q4[["end", "val", "form", "filed"]]], ignore_index=True)
    else:
        q = q[["end", "val", "form", "filed"]].copy()

    q = _dedupe_end(q)
    return q.sort_values(["end", "filed"]).groupby("end")["val"].last().sort_index()

# -----------------------------
# instant (BS): end 기준 최신 1개
# -----------------------------
def _build_series_instant(df: pd.DataFrame) -> pd.Series:
    if df.empty:
        return pd.Series(dtype="float")
    df = _dedupe_end_fy_fp(df)
    df = _dedupe_end(df)
    return df.sort_values(["end", "filed"]).groupby("end")["val"].last().sort_index()

# -----------------------------
# 날짜 정렬(선택): fiscal end -> calendar quarter end
# -----------------------------
def _align_to_calendar_quarter_end(idx: pd.DatetimeIndex) -> pd.DatetimeIndex:
    # 예: 2024-12-28(토) -> 2024-12-31
    return idx.to_period("Q").to_timestamp("Q")

# -----------------------------
# main
# -----------------------------
def build_quarterly_financials(
    ticker: str,
    user_agent: str,
    tags_map: Optional[Dict[str, List[str]]] = None,
    start_date: Optional[str] = None,
    align_to_calendar_qend: bool = False,
    debug: bool = False,
) -> pd.DataFrame:
    tags_map = tags_map or DEFAULT_TAGS
    cf = fetch_companyfacts(ticker, user_agent=user_agent)

    series: Dict[str, pd.Series] = {}

    for col, tag_candidates in tags_map.items():
        raw = extract_best_concept(cf, tag_candidates)
        raw = _prep_core(raw)

        if raw.empty:
            series[col] = pd.Series(dtype="float")
            if debug:
                print(f"[{ticker}] {col}: EMPTY")
            continue

        # duration / instant 분기
        if col in DURATION_COLS:
            s = _build_quarter_series_duration(raw)
        else:
            s = _build_series_instant(raw)

        # 날짜 정렬 옵션
        if align_to_calendar_qend and len(s.index) > 0:
            new_idx = _align_to_calendar_quarter_end(pd.DatetimeIndex(s.index))
            s = pd.Series(s.values, index=new_idx).sort_index()
            # 같은 달력 분기말로 겹치면 마지막 값 채택
            s = s.groupby(level=0).last()

        series[col] = s

        if debug:
            tag_used = raw["tag"].iloc[0] if "tag" in raw.columns and len(raw) else "NA"
            print(f"[{ticker}] {col}: tag={tag_used}, points={len(series[col])}")

    out = pd.DataFrame(series).sort_index()

    if start_date is not None:
        out = out.loc[pd.to_datetime(start_date):]

    return out

# -----------------------------
# Example
# -----------------------------
if __name__ == "__main__":
    USER_AGENT = "HoyoungPark Research (stox1224@email.com)"  # 본인 것으로
    df = build_quarterly_financials(
        "AAPL",
        user_agent=USER_AGENT,
        start_date="2020-01-01",
        align_to_calendar_qend=True,  # ✅ 당신 스샷처럼 03/31,06/30,09/30,12/31로 맞추려면 True
        debug=True
    )
    print(df.tail(12))


[AAPL] revenue: tag=SalesRevenueNet, points=39
[AAPL] net_income: tag=NetIncomeLoss, points=64
[AAPL] operating_income: tag=OperatingIncomeLoss, points=53
[AAPL] total_assets: tag=Assets, points=66
[AAPL] current_assets: tag=AssetsCurrent, points=66
[AAPL] total_liabilities: tag=Liabilities, points=66
[AAPL] current_liabilities: tag=LiabilitiesCurrent, points=66
[AAPL] equity: tag=StockholdersEquity, points=68
            revenue    net_income  operating_income  total_assets  \
end                                                                 
2022-12-31      NaN  2.999800e+10      3.601600e+10  3.467470e+11   
2023-06-30      NaN  2.416000e+10      2.831800e+10  3.321600e+11   
2023-09-30      NaN  1.988100e+10      2.299800e+10  3.525830e+11   
2023-12-31      NaN  3.391600e+10      4.037300e+10  3.535140e+11   
2024-03-31      NaN  2.363600e+10      2.790000e+10  3.374110e+11   
2024-06-30      NaN  2.144800e+10      2.535200e+10  3.316120e+11   
2024-09-30      NaN           NaN 

In [6]:
df

,revenue,net_income,operating_income,total_assets,current_assets,total_liabilities,current_liabilities,equity
end,,,,,,,,
2020-03-28,NaN,1.124900e+10,1.285300e+10,3.204000e+11,1.437530e+11,2.419750e+11,9.609400e+10,7.842500e+10
2020-06-27,NaN,1.125300e+10,1.309100e+10,3.173440e+11,1.400650e+11,2.450620e+11,9.531800e+10,7.228200e+10
2020-09-26,NaN,-1.671800e+10,-1.887500e+10,3.238880e+11,1.437130e+11,2.585490e+11,1.053920e+11,6.533900e+10
2020-12-26,NaN,2.875500e+10,3.353400e+10,3.540540e+11,1.541060e+11,2.878300e+11,1.325070e+11,6.622400e+10
2021-03-27,NaN,2.363000e+10,2.750300e+10,3.371580e+11,1.214650e+11,2.679800e+11,1.063850e+11,6.917800e+10
2021-06-26,NaN,2.174400e+10,2.412600e+10,3.298400e+11,1.144230e+11,2.655600e+11,1.077540e+11,6.428000e+10
2021-09-25,NaN,1.559800e+10,1.440600e+10,3.510020e+11,1.348360e+11,2.879120e+11,1.254810e+11,6.309000e+10
2021-12-25,NaN,3.463000e+10,4.148800e+10,3.811910e+11,1.531540e+11,3.092590e+11,1.475740e+11,7.193200e+10
2022-03-26,NaN,2.501000e+10,2.997900e+10,3.506620e+11,1.181800e+11,2.832630e+11,1.275080e+11,6.739900e+10


In [8]:
def debug_revenue_tags(ticker: str, user_agent: str):
    cf = fetch_companyfacts(ticker, user_agent=user_agent)
    gaap = cf.get("facts", {}).get("us-gaap", {})
    keys = sorted(gaap.keys())

    cand = [k for k in keys if ("Revenue" in k) or ("Sales" in k) or ("Revenues" in k)]
    print(f"Revenue-like tags count = {len(cand)}")
    print(cand[:50])  # 처음 50개만

    # 각 태그별로 USD unit 존재/관측치 수 출력
    rows = []
    for tag in cand:
        units = gaap[tag].get("units", {})
        usd_n = len(units.get("USD", []))
        any_units = list(units.keys())[:5]
        rows.append((tag, usd_n, any_units))
    df = pd.DataFrame(rows, columns=["tag", "usd_points", "sample_units"]).sort_values("usd_points", ascending=False)
    print(df.head(20).to_string(index=False))

# 사용 예
debug_revenue_tags("AAPL", USER_AGENT)

Revenue-like tags count = 8
['ContractWithCustomerLiabilityRevenueRecognized', 'DeferredRevenueCurrent', 'DeferredRevenueNoncurrent', 'IncreaseDecreaseInDeferredRevenue', 'RevenueFromContractWithCustomerExcludingAssessedTax', 'Revenues', 'SalesRevenueNet', 'SalesRevenueServicesGross']
                                                tag  usd_points sample_units
                                    SalesRevenueNet         210        [USD]
RevenueFromContractWithCustomerExcludingAssessedTax         109        [USD]
                  IncreaseDecreaseInDeferredRevenue          95        [USD]
     ContractWithCustomerLiabilityRevenueRecognized          93        [USD]
                             DeferredRevenueCurrent          82        [USD]
                          DeferredRevenueNoncurrent          82        [USD]
                                           Revenues          11        [USD]
                          SalesRevenueServicesGross           1        [USD]


In [16]:
import time
import requests
import pandas as pd
from typing import Dict, List, Optional, Tuple

SEC_TICKER_CIK_URL = "https://www.sec.gov/files/company_tickers.json"
SEC_COMPANYFACTS_URL = "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json"

FORM_PRIORITY = {"10-Q": 4, "10-K": 3, "10-Q/A": 2, "10-K/A": 1}
KEEP_FORMS = set(FORM_PRIORITY.keys())

DURATION_COLS = {"revenue", "net_income", "operating_income"}

DEFAULT_TAGS = {
    # ✅ AAPL: SalesRevenueNet가 가장 풍부 (당신 디버그 결과)
    "revenue": [
        "SalesRevenueNet",
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "Revenue",
        "Revenues",
    ],
    "net_income": ["NetIncomeLoss", "ProfitLoss"],
    "operating_income": ["OperatingIncomeLoss"],

    "total_assets": ["Assets"],
    "current_assets": ["AssetsCurrent"],
    "total_liabilities": ["Liabilities"],
    "current_liabilities": ["LiabilitiesCurrent"],
    "equity": [
        "StockholdersEquity",
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    ],
}

# -----------------------------
# HTTP
# -----------------------------
def sec_get_json(url: str, user_agent: str, timeout: int = 30, sleep_sec: float = 0.12) -> dict:
    headers = {
        "User-Agent": user_agent,
        "Accept": "application/json",
        "Accept-Encoding": "gzip, deflate",
    }
    r = requests.get(url, headers=headers, timeout=timeout)
    r.raise_for_status()
    if sleep_sec:
        time.sleep(sleep_sec)
    return r.json()

def ticker_to_cik10(ticker: str, user_agent: str) -> str:
    data = sec_get_json(SEC_TICKER_CIK_URL, user_agent=user_agent)
    t = ticker.upper().strip()
    for _, v in data.items():
        if str(v.get("ticker", "")).upper() == t:
            return f"{int(v['cik_str']):010d}"
    raise ValueError(f"CIK not found for ticker={ticker}")

def fetch_companyfacts(ticker: str, user_agent: str) -> dict:
    cik10 = ticker_to_cik10(ticker, user_agent=user_agent)
    return sec_get_json(SEC_COMPANYFACTS_URL.format(cik10=cik10), user_agent=user_agent)

# -----------------------------
# parsing
# -----------------------------
def _pick_unit_block(concept_obj: dict) -> Optional[Tuple[str, List[dict]]]:
    units = concept_obj.get("units", {})
    if not units:
        return None
    if "USD" in units:
        return ("USD", units["USD"])
    best_unit = max(units.keys(), key=lambda k: len(units.get(k, [])))
    return (best_unit, units[best_unit])

def _to_df(items: List[dict]) -> pd.DataFrame:
    df = pd.DataFrame(items).copy()
    if df.empty:
        return df
    for col in ["end", "start", "filed"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    if "val" in df.columns:
        df["val"] = pd.to_numeric(df["val"], errors="coerce")
    return df

def extract_concept(companyfacts: dict, tag_candidates: List[str]) -> pd.DataFrame:
    facts = companyfacts.get("facts", {}).get("us-gaap", {})
    for tag in tag_candidates:
        if tag not in facts:
            continue
        unit_block = _pick_unit_block(facts[tag])
        if not unit_block:
            continue
        unit, items = unit_block
        df = _to_df(items)
        if df.empty:
            continue
        df = df.dropna(subset=["end", "val"]).copy()
        df["tag"] = tag
        df["unit"] = unit
        return df
    return pd.DataFrame()

# -----------------------------
# cleaning (revenue is lenient)
# -----------------------------
def _keep_forms_if_possible(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or "form" not in df.columns:
        return df
    return df[df["form"].isin(KEEP_FORMS)].copy()

def _prep_generic(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    need = {"end", "val", "form", "filed"}
    if not need.issubset(df.columns):
        return pd.DataFrame()
    df = df.dropna(subset=list(need)).copy()
    df = _keep_forms_if_possible(df)
    return df

def _prep_revenue_lenient(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    need = {"end", "val"}
    if not need.issubset(df.columns):
        return pd.DataFrame()
    df = df.dropna(subset=list(need)).copy()
    df = _keep_forms_if_possible(df)
    return df

# -----------------------------
# dedupe
# -----------------------------
def _add_form_score(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()
    if "form" in df.columns:
        df["form_score"] = df["form"].map(FORM_PRIORITY).fillna(-1).astype(int)
    else:
        df["form_score"] = -1
    return df

def _dedupe_end_fy_fp(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = _add_form_score(df)
    if not all(k in df.columns for k in ["end", "fy", "fp"]):
        return df
    df["filed_sort"] = df["filed"] if "filed" in df.columns else pd.NaT
    df["filed_sort"] = df["filed_sort"].fillna(pd.Timestamp.min)
    df = df.sort_values(["end", "fy", "fp", "form_score", "filed_sort"],
                        ascending=[True, True, True, False, False])
    df = df.drop_duplicates(subset=["end", "fy", "fp"], keep="first")
    return df.drop(columns=["filed_sort"])

def _dedupe_end(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = _add_form_score(df)
    df["filed_sort"] = df["filed"] if "filed" in df.columns else pd.NaT
    df["filed_sort"] = df["filed_sort"].fillna(pd.Timestamp.min)
    df = df.sort_values(["end", "form_score", "filed_sort"], ascending=[True, False, False])
    df = df.drop_duplicates(subset=["end"], keep="first")
    return df.drop(columns=["filed_sort"])

# -----------------------------
# duration: quarter first else YTD diff
# -----------------------------
def _duration_days(df: pd.DataFrame) -> pd.Series:
    if "start" not in df.columns:
        return pd.Series([pd.NA] * len(df), index=df.index)
    return (df["end"] - df["start"]).dt.days

def _select_quarter_like(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    out["dur_days"] = _duration_days(out)
    return out[(out["dur_days"].notna()) & (out["dur_days"].between(70, 120))].copy()

def _build_duration_series(df: pd.DataFrame) -> pd.Series:
    if df.empty:
        return pd.Series(dtype="float")

    if not {"fy", "fp"}.issubset(df.columns):
        df2 = _dedupe_end(df)
        return df2.sort_values("end").set_index("end")["val"]

    df = df.copy()
    df["fy"] = pd.to_numeric(df["fy"], errors="coerce")
    df = df.dropna(subset=["fy", "fp"])
    df["fy"] = df["fy"].astype(int)

    qlike = _select_quarter_like(df)
    if not qlike.empty:
        qlike = _dedupe_end_fy_fp(qlike)
        qlike = _dedupe_end(qlike)
        return qlike.sort_values("end").set_index("end")["val"]

    y = _dedupe_end_fy_fp(df)
    q = y[y["fp"].isin(["Q1", "Q2", "Q3"])].copy()
    fy = y[y["fp"].isin(["FY"])].copy()

    if q.empty:
        y2 = _dedupe_end(y)
        return y2.sort_values("end").set_index("end")["val"]

    q = q.sort_values(["fy", "end"])
    q["qval"] = q.groupby("fy")["val"].diff()
    q["qval"] = q["qval"].fillna(q["val"])
    q["val"] = q["qval"]
    q = q.drop(columns=["qval"])

    if not fy.empty:
        fy2 = fy[["fy", "end", "val"]].rename(columns={"val": "fy_val"})
        ytd_q3 = y[y["fp"] == "Q3"][["fy", "val"]].rename(columns={"val": "ytd_q3"})
        merged = fy2.merge(ytd_q3, on="fy", how="left")

        qsum = q.groupby("fy")["val"].sum().rename("qsum").reset_index()
        merged = merged.merge(qsum, on="fy", how="left")

        merged["q4"] = merged["fy_val"] - merged["ytd_q3"]
        merged.loc[merged["q4"].isna(), "q4"] = merged["fy_val"] - merged["qsum"]

        q4 = merged[["end", "q4"]].copy()
        q4["val"] = q4["q4"]
        q4 = q4.drop(columns=["q4"])

        qq = pd.concat([q[["end", "val"]], q4[["end", "val"]]], ignore_index=True)
    else:
        qq = q[["end", "val"]].copy()

    qq = _dedupe_end(qq)
    return qq.sort_values("end").set_index("end")["val"]

def _build_instant_series(df: pd.DataFrame) -> pd.Series:
    df = _dedupe_end_fy_fp(df)
    df = _dedupe_end(df)
    return df.sort_values("end").set_index("end")["val"]

# -----------------------------
# ✅ 핵심: end를 달력 분기말로 통일
# -----------------------------
def _to_calendar_qend_index(idx: pd.DatetimeIndex) -> pd.DatetimeIndex:
    return idx.to_period("Q").to_timestamp("Q")

def _nearest_calendar_qend(ts: pd.Timestamp) -> pd.Timestamp:
    """
    ts를 기준으로 '이전 분기말'과 '해당 분기말' 중 더 가까운 날짜로 매핑.
    (AAPL처럼 토요일/일요일 분기말이 많은 기업에 특히 중요)
    """
    p = ts.to_period("Q")
    prev_qend = (p.start_time - pd.Timedelta(days=1)).normalize()  # 직전 분기말
    this_qend = p.end_time.normalize()                             # 해당 분기말

    if abs(ts - prev_qend) <= abs(this_qend - ts):
        return prev_qend
    return this_qend


def _normalize_series_index_to_calendar_qend(s: pd.Series) -> pd.Series:
    if s.empty:
        return s
    new_idx = pd.DatetimeIndex([_nearest_calendar_qend(pd.Timestamp(x)) for x in s.index])
    out = pd.Series(s.values, index=new_idx).sort_index()
    # 같은 달력 분기말로 겹치면 마지막 값 채택
    return out.groupby(level=0).last()

# -----------------------------
# main
# -----------------------------
def build_quarterly_financials(
    ticker: str,
    user_agent: str,
    tags_map: Optional[Dict[str, List[str]]] = None,
    start_date: Optional[str] = None,
    align_to_calendar_qend: bool = True,   # ✅ 기본 True로 (당신 표 형태/병합 안정성)
    debug: bool = False,
) -> pd.DataFrame:
    tags_map = tags_map or DEFAULT_TAGS
    cf = fetch_companyfacts(ticker, user_agent=user_agent)

    series: Dict[str, pd.Series] = {}

    for col, tag_candidates in tags_map.items():
        raw = extract_concept(cf, tag_candidates)
        if raw.empty:
            series[col] = pd.Series(dtype="float")
            if debug:
                print(f"[{ticker}] {col}: EMPTY (no tag matched)")
            continue

        df = _prep_revenue_lenient(raw) if col == "revenue" else _prep_generic(raw)
        if df.empty:
            series[col] = pd.Series(dtype="float")
            if debug:
                print(f"[{ticker}] {col}: EMPTY (filtered out) tag={raw['tag'].iloc[0]}")
            continue

        s = _build_duration_series(df) if col in DURATION_COLS else _build_instant_series(df)

        # ✅ index 정규화로 NaN 문제 해결
        if align_to_calendar_qend:
            s = _normalize_series_index_to_calendar_qend(s)

        series[col] = s

        if debug:
            tag_used = raw["tag"].iloc[0] if "tag" in raw.columns else "NA"
            print(f"[{ticker}] {col}: tag={tag_used}, points={len(s)}")

    out = pd.DataFrame(series).sort_index()
    if start_date is not None:
        out = out.loc[pd.to_datetime(start_date):]
    return out

# Example
# if __name__ == "__main__":
#     USER_AGENT = "HoyoungPark Research (stox1224@email.com)"
#     df = build_quarterly_financials("AAPL", user_agent=USER_AGENT, start_date="2022-01-01", debug=True)
#     print(df.tail(12))


In [17]:
USER_AGENT = "HoyoungPark Research (stox1224@email.com)"
df = build_quarterly_financials("AAPL", user_agent=USER_AGENT, start_date="2022-01-01", debug=True)
print(df.tail(12))

[AAPL] revenue: tag=SalesRevenueNet, points=40
[AAPL] net_income: tag=NetIncomeLoss, points=65
[AAPL] operating_income: tag=OperatingIncomeLoss, points=53
[AAPL] total_assets: tag=Assets, points=68
[AAPL] current_assets: tag=AssetsCurrent, points=68
[AAPL] total_liabilities: tag=Liabilities, points=68
[AAPL] current_liabilities: tag=LiabilitiesCurrent, points=68
[AAPL] equity: tag=StockholdersEquity, points=70
            revenue    net_income  operating_income  total_assets  \
2023-03-31      NaN  2.416000e+10      2.831800e+10  3.321600e+11   
2023-06-30      NaN  1.988100e+10      2.299800e+10  3.350380e+11   
2023-09-30      NaN           NaN               NaN  3.525830e+11   
2023-12-31      NaN  3.391600e+10      4.037300e+10  3.535140e+11   
2024-03-31      NaN  2.363600e+10      2.790000e+10  3.374110e+11   
2024-06-30      NaN  2.144800e+10      2.535200e+10  3.316120e+11   
2024-09-30      NaN           NaN               NaN  3.649800e+11   
2024-12-31      NaN  3.633000e+10 

In [11]:
df

,revenue,net_income,operating_income,total_assets,current_assets,total_liabilities,current_liabilities,equity
end,,,,,,,,
2022-03-26,NaN,2.501000e+10,2.997900e+10,3.506620e+11,1.181800e+11,2.832630e+11,1.275080e+11,6.739900e+10
2022-06-25,NaN,1.944200e+10,2.307600e+10,3.363090e+11,1.122920e+11,2.782020e+11,1.298730e+11,5.810700e+10
2022-09-24,NaN,NaN,NaN,3.527550e+11,1.354050e+11,3.020830e+11,1.539820e+11,5.067200e+10
2022-12-31,NaN,2.999800e+10,3.601600e+10,3.467470e+11,1.287770e+11,2.900200e+11,1.372860e+11,5.672700e+10
2023-04-01,NaN,2.416000e+10,2.831800e+10,3.321600e+11,1.129130e+11,2.700020e+11,1.200750e+11,6.215800e+10
2023-07-01,NaN,1.988100e+10,2.299800e+10,3.350380e+11,1.226590e+11,2.747640e+11,1.249630e+11,6.027400e+10
2023-09-30,NaN,NaN,NaN,3.525830e+11,1.435660e+11,2.904370e+11,1.453080e+11,6.214600e+10
2023-12-30,NaN,3.391600e+10,4.037300e+10,3.535140e+11,1.436920e+11,2.794140e+11,1.339730e+11,7.410000e+10
2024-03-30,NaN,2.363600e+10,2.790000e+10,3.374110e+11,1.284160e+11,2.632170e+11,1.238220e+11,7.419400e+10
